# Per-Dataset State Accuracy — độc lập hoàn toàn

Không phụ thuộc `Evaluation_Capability.ipynb`. Chỉ nạp checkpoint Pha 2 (bỏ Pha 1 — không
cần cho accuracy), chỉ suy luận trên TEST (bỏ VAL, bỏ UMAP/Capability A–F). Dùng
`deterministic_crop=True` + `model.eval()` — kết quả tất định, chạy lại nhiều lần cho
cùng một số.

**Kiểm tra chéo bắt buộc**: cột `accuracy` ở hàng `ALL` phải khớp 82,31% / 89,88% / 87,64%
(Activation/Tension/Stability) — đúng số đã có trong báo cáo (Bảng 3.11). Nếu lệch, dừng
lại kiểm tra checkpoint/đường dẫn trước khi tin bảng theo dataset bên dưới.

In [ ]:
!pip install -q torch numpy pandas tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json, math, random
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, f1_score

DRIVE_ROOT = Path("/content/drive/MyDrive")
PHASE2_CKPT = DRIVE_ROOT / "motionemo_v10_phase2/checkpoints/best_phase2_state.pt"

LOCAL_DATA_ROOT = Path("/content/processed_v10_safe")
DRIVE_DATA_ROOT = DRIVE_ROOT / "data/processed_v10_safe"
DATA_ROOT = LOCAL_DATA_ROOT if (LOCAL_DATA_ROOT / "all_metadata.csv").exists() else DRIVE_DATA_ROOT
METADATA_PATH = DATA_ROOT / "all_metadata.csv"
PRECOMPUTED_DIR = DATA_ROOT / "precomputed"

OUTPUT_ROOT = DRIVE_ROOT / "motionemo_v10_eval_capability"
OUT_DIR = OUTPUT_ROOT / "per_dataset_accuracy"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = DEVICE == "cuda"
STATE_NAMES = ["activation", "tension", "stability"]

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

def autocast_context():
    from contextlib import nullcontext
    return torch.amp.autocast("cuda", enabled=True) if (AMP_ENABLED and DEVICE == "cuda") else nullcontext()

print(f"Device: {DEVICE} | Metadata exists: {METADATA_PATH.exists()} | Checkpoint exists: {PHASE2_CKPT.exists()}")

Device: cpu | Metadata exists: True | Checkpoint exists: True


In [ ]:
# Kiến trúc model — sao chép NGUYÊN VĂN từ Training_phase2 / Evaluation_Capability, không sửa
def make_padding_mask(lengths, max_len):
    idx = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)

class MotionEmoBackboneV10(nn.Module):
    def __init__(self, n_mels=80, embed_dim=512, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.prosody_encoder = nn.Sequential(
            nn.Conv1d(3, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.mel_encoder = nn.Sequential(
            nn.Conv1d(n_mels, 256, kernel_size=5, padding=2), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.fusion_proj = nn.Sequential(nn.Linear(512, embed_dim), nn.LayerNorm(embed_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.attn_pool = nn.Sequential(nn.Linear(embed_dim, embed_dim // 4), nn.Tanh(), nn.Linear(embed_dim // 4, 1))

    def forward(self, mel, f0, energy, srate, lengths=None):
        B, _, T = mel.shape
        prosody_in = torch.stack([f0, energy, srate], dim=1)
        p_feat = self.prosody_encoder(prosody_in)
        m_feat = self.mel_encoder(mel)
        fused = torch.cat([p_feat, m_feat], dim=1)
        x = self.fusion_proj(fused.transpose(1, 2))
        pad_mask = make_padding_mask(lengths, T) if lengths is not None else None
        H_frame = self.transformer(x, src_key_padding_mask=pad_mask)
        attn_logits = self.attn_pool(H_frame).squeeze(-1)
        if pad_mask is not None:
            attn_logits = attn_logits.masked_fill(pad_mask, -1e4)
        attn = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        return H_frame, (attn * H_frame).sum(dim=1)

class StateHead(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, H_utt):
        return self.net(H_utt)

print("Kiến trúc model đã định nghĩa.")

Kiến trúc model đã định nghĩa.


In [ ]:
if not PHASE2_CKPT.exists():
    raise FileNotFoundError(f"Thiếu checkpoint Phase 2: {PHASE2_CKPT}")

phase2_ckpt = torch.load(PHASE2_CKPT, map_location=DEVICE, weights_only=False)
cfg = phase2_ckpt.get("cfg", {})
mc = cfg.get("model", {}) if isinstance(cfg, dict) else {}
embed_dim = int(mc.get("embedding_dim", 512))
state_hidden = int(mc.get("state_hidden", 256))
num_state_classes = int(mc.get("num_state_classes", 3))

phase2_backbone = MotionEmoBackboneV10(
    n_mels=80, embed_dim=embed_dim, num_heads=int(mc.get("num_heads", 8)),
    num_layers=int(mc.get("num_layers", 6)), dropout=float(mc.get("dropout", 0.1)),
).to(DEVICE)
phase2_backbone.load_state_dict(phase2_ckpt["backbone"], strict=True)
phase2_backbone.eval()

activation_head = StateHead(embed_dim, state_hidden, num_state_classes).to(DEVICE)
tension_head    = StateHead(embed_dim, state_hidden, num_state_classes).to(DEVICE)
stability_head  = StateHead(embed_dim, state_hidden, num_state_classes).to(DEVICE)
activation_head.load_state_dict(phase2_ckpt["activation_head"]); activation_head.eval()
tension_head.load_state_dict(phase2_ckpt["tension_head"]);       tension_head.eval()
stability_head.load_state_dict(phase2_ckpt["stability_head"]);   stability_head.eval()

# state_thresholds PHẢI lấy từ checkpoint, không tính lại — khớp đúng lúc train.
STATE_THRESHOLDS = phase2_ckpt.get("state_thresholds")
if STATE_THRESHOLDS is None:
    raise RuntimeError("Checkpoint không có 'state_thresholds'.")

print(f"epoch={phase2_ckpt.get('epoch')} | best_state_acc={phase2_ckpt.get('best_state_acc')}")
print(f"state_thresholds={STATE_THRESHOLDS}")

epoch=25 | best_state_acc=0.868068068068068
state_thresholds={'activation': [-1.1789047548920202, -1.0910444258413774], 'tension': [0.5080785265271103, 0.608056457961331], 'stability': [-0.41222898640634226, -0.33455883524576263]}


/tmp/ipykernel_3450/4271228891.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# Nhãn proxy + Dataset — sao chép NGUYÊN VĂN từ Training_phase2 / Evaluation_Capability
def _to_np(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def compute_vocal_state_scores_from_arrays(f0_raw, energy_raw, zcr_raw, voiced_mask):
    f0 = _to_np(f0_raw).astype(np.float32).reshape(-1)
    energy = _to_np(energy_raw).astype(np.float32).reshape(-1)
    zcr = _to_np(zcr_raw).astype(np.float32).reshape(-1)
    voiced = _to_np(voiced_mask).astype(np.float32).reshape(-1) > 0.5

    T = min(len(f0), len(energy), len(zcr), len(voiced))
    if T <= 2:
        return {"activation": 0.0, "tension": 0.0, "stability": 0.0}

    f0 = f0[:T]
    energy = np.maximum(energy[:T], 0.0)
    zcr = np.maximum(zcr[:T], 0.0)
    voiced = voiced[:T] & np.isfinite(f0) & (f0 > 0)
    eps = 1e-8

    e_mean = float(np.mean(energy)); e_std = float(np.std(energy))
    e_p95 = float(np.percentile(energy, 95)) + eps
    e_cv = e_std / (e_mean + eps)
    pause_ratio = float(np.mean((energy / e_p95) < 0.03))

    z_mean = float(np.mean(zcr)); z_std = float(np.std(zcr))

    if voiced.sum() > 2:
        vf0 = f0[voiced]
        f0_mean = float(np.mean(vf0)) + eps
        f0_range_rel = (float(np.percentile(vf0, 90)) - float(np.percentile(vf0, 10))) / f0_mean
        f0_motion_rel = float(np.std(np.diff(vf0))) / f0_mean
    else:
        f0_range_rel = 0.0
        f0_motion_rel = 0.0

    log_energy = math.log(e_mean + 1e-6)
    activation = 0.45 * log_energy + 0.35 * z_mean + 0.20 * f0_range_rel
    tension = 0.45 * f0_motion_rel + 0.35 * e_cv + 0.20 * z_std
    instability = 0.45 * pause_ratio + 0.35 * f0_motion_rel + 0.20 * e_cv

    return {"activation": float(activation), "tension": float(tension), "stability": float(-instability)}

def assign_state_label(score, thresholds):
    lo, hi = thresholds
    if score <= lo: return 0
    if score <= hi: return 1
    return 2

class Phase2StateDataset(Dataset):
    def __init__(self, df_data, thresholds, max_frames=500, deterministic_crop=True):
        self.df = df_data.reset_index(drop=True)
        self.thresholds = thresholds
        self.max_frames = max_frames
        self.deterministic_crop = deterministic_crop

    def __len__(self):
        return len(self.df)

    def _crop_start(self, T):
        if T <= self.max_frames: return 0
        if self.deterministic_crop: return max(0, (T - self.max_frames) // 2)
        return random.randint(0, T - self.max_frames)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        obj = torch.load(Path(row["feature_path"]), map_location="cpu", weights_only=False)

        mel = obj["mel"].float(); f0 = obj["f0"].float()
        energy = obj["energy"].float(); srate = obj["speaking_rate"].float()
        f0_raw = obj["f0_raw"].float(); energy_raw = obj["energy_raw"].float()
        zcr_raw = obj["zcr_raw"].float(); voiced_mask = obj["voiced_mask"].float()

        T = mel.shape[1]
        if T > self.max_frames:
            start = self._crop_start(T); end = start + self.max_frames
            mel = mel[:, start:end]; f0 = f0[start:end]; energy = energy[start:end]; srate = srate[start:end]
            f0_raw = f0_raw[start:end]; energy_raw = energy_raw[start:end]
            zcr_raw = zcr_raw[start:end]; voiced_mask = voiced_mask[start:end]

        scores = compute_vocal_state_scores_from_arrays(f0_raw, energy_raw, zcr_raw, voiced_mask)
        return dict(
            mel=mel, f0=f0, energy=energy, srate=srate, length=mel.shape[1],
            activation_label=assign_state_label(scores["activation"], self.thresholds["activation"]),
            tension_label=assign_state_label(scores["tension"], self.thresholds["tension"]),
            stability_label=assign_state_label(scores["stability"], self.thresholds["stability"]),
            sample_id=row["sample_id"], dataset=row["dataset"],
        )

def collate_state_fn(batch):
    max_len = max(item["length"] for item in batch)
    mels, f0s, energies, srates, lengths = [], [], [], [], []
    act, ten, sta, sample_ids, datasets = [], [], [], [], []
    for item in batch:
        pad = max_len - item["length"]
        mels.append(F.pad(item["mel"], (0, pad)).unsqueeze(0))
        f0s.append(F.pad(item["f0"], (0, pad)).unsqueeze(0))
        energies.append(F.pad(item["energy"], (0, pad)).unsqueeze(0))
        srates.append(F.pad(item["srate"], (0, pad)).unsqueeze(0))
        lengths.append(item["length"])
        act.append(item["activation_label"]); ten.append(item["tension_label"]); sta.append(item["stability_label"])
        sample_ids.append(item["sample_id"]); datasets.append(item["dataset"])
    return dict(
        mel=torch.cat(mels, dim=0), f0=torch.cat(f0s, dim=0), energy=torch.cat(energies, dim=0), srate=torch.cat(srates, dim=0),
        lengths=torch.tensor(lengths, dtype=torch.long),
        activation_label=torch.tensor(act, dtype=torch.long), tension_label=torch.tensor(ten, dtype=torch.long),
        stability_label=torch.tensor(sta, dtype=torch.long), sample_id=sample_ids, dataset=datasets,
    )

print("Đã định nghĩa hàm nhãn proxy + Dataset.")

Đã định nghĩa hàm nhãn proxy + Dataset.


In [ ]:
df_meta = pd.read_csv(METADATA_PATH)
df_meta["feature_path"] = df_meta["feature_path"].apply(lambda p: str(PRECOMPUTED_DIR / Path(str(p)).name))
df_meta = df_meta[df_meta["feature_path"].apply(lambda p: Path(p).exists())].copy()

test_df = df_meta[df_meta["split"] == "test"].reset_index(drop=True)
test_ds = Phase2StateDataset(test_df, STATE_THRESHOLDS, max_frames=500, deterministic_crop=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_state_fn,
                          num_workers=2, pin_memory=(DEVICE == "cuda"))
print(f"TEST: {len(test_ds)} mẫu, {len(test_loader)} batch")

TEST: 1651 mẫu, 104 batch


In [ ]:
@torch.no_grad()
def extract_state_predictions(loader):
    y_true = {s: [] for s in STATE_NAMES}
    y_pred = {s: [] for s in STATE_NAMES}
    sample_ids, datasets = [], []

    for batch in tqdm(loader, desc="Suy luận test"):
        mel = batch["mel"].to(DEVICE); f0 = batch["f0"].to(DEVICE)
        energy = batch["energy"].to(DEVICE); srate = batch["srate"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)

        with autocast_context():
            _, h = phase2_backbone(mel, f0, energy, srate, lengths=lengths)
            ap = F.softmax(activation_head(h), dim=-1)
            tp = F.softmax(tension_head(h), dim=-1)
            sp = F.softmax(stability_head(h), dim=-1)

        y_true["activation"].extend(batch["activation_label"].numpy().tolist())
        y_true["tension"].extend(batch["tension_label"].numpy().tolist())
        y_true["stability"].extend(batch["stability_label"].numpy().tolist())
        y_pred["activation"].extend(ap.float().cpu().numpy().argmax(axis=-1).tolist())
        y_pred["tension"].extend(tp.float().cpu().numpy().argmax(axis=-1).tolist())
        y_pred["stability"].extend(sp.float().cpu().numpy().argmax(axis=-1).tolist())
        sample_ids.extend(batch["sample_id"]); datasets.extend(batch["dataset"])

    for s in STATE_NAMES:
        y_true[s] = np.array(y_true[s], dtype=np.int64)
        y_pred[s] = np.array(y_pred[s], dtype=np.int64)
    return dict(y_true=y_true, y_pred=y_pred, sample_id=sample_ids, dataset=np.array(datasets))

test_out = extract_state_predictions(test_loader)
print("Suy luận xong.")

Suy luận test:   0%|          | 0/104 [00:00<?, ?it/s]

Suy luận xong.


In [ ]:
def bootstrap_acc_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        accs[b] = (y_true[idx] == y_pred[idx]).mean()
    return float(accs.mean()), (float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5)))

EXPECTED_ALL_ACC = {"activation": 0.8231, "tension": 0.8988, "stability": 0.8764}

rows = []
for state in STATE_NAMES:
    yt_all, yp_all = test_out["y_true"][state], test_out["y_pred"][state]
    acc, ci = bootstrap_acc_ci(yt_all, yp_all)
    flag = "" if abs(acc - EXPECTED_ALL_ACC[state]) < 0.01 else "  <-- LỆCH SO VỚI BÁO CÁO, KIỂM TRA LẠI"
    print(f"[ALL] {state}: accuracy={acc:.4f} (kỳ vọng ~{EXPECTED_ALL_ACC[state]:.4f}){flag}")
    rows.append({"dataset": "ALL", "state": state, "n": len(yt_all), "accuracy": acc,
                 "ci_low": ci[0], "ci_high": ci[1],
                 "balanced_accuracy": balanced_accuracy_score(yt_all, yp_all),
                 "macro_f1": f1_score(yt_all, yp_all, average="macro")})

    for ds in sorted(set(test_out["dataset"])):
        mask = test_out["dataset"] == ds
        yt, yp = yt_all[mask], yp_all[mask]
        if len(yt) < 10 or len(np.unique(yt)) < 2:
            rows.append({"dataset": ds, "state": state, "n": int(mask.sum()), "accuracy": np.nan,
                         "ci_low": np.nan, "ci_high": np.nan, "balanced_accuracy": np.nan, "macro_f1": np.nan})
            continue
        acc, ci = bootstrap_acc_ci(yt, yp)
        rows.append({"dataset": ds, "state": state, "n": len(yt), "accuracy": acc,
                     "ci_low": ci[0], "ci_high": ci[1],
                     "balanced_accuracy": balanced_accuracy_score(yt, yp),
                     "macro_f1": f1_score(yt, yp, average="macro")})

per_dataset_df = pd.DataFrame(rows)
per_dataset_df.to_csv(OUT_DIR / "per_dataset_accuracy.csv", index=False)
print(per_dataset_df.to_string(index=False))

[ALL] activation: accuracy=0.8225 (kỳ vọng ~0.8231)
[ALL] tension: accuracy=0.8984 (kỳ vọng ~0.8988)
[ALL] stability: accuracy=0.8762 (kỳ vọng ~0.8764)
dataset      state    n  accuracy   ci_low  ci_high  balanced_accuracy  macro_f1
    ALL activation 1651  0.822502 0.804952 0.840703           0.824026  0.824207
  crema activation  947  0.832058 0.807788 0.855333           0.836795  0.833727
iemocap activation  560  0.769821 0.737455 0.803571           0.777676  0.774806
ravdess activation  144  0.964444 0.930556 0.993056           0.661836  0.660650
    ALL    tension 1651  0.898354 0.884313 0.912174           0.897337  0.893254
  crema    tension  947  0.893532 0.873258 0.913411           0.900547  0.892451
iemocap    tension  560  0.885681 0.858929 0.910714           0.871373  0.861176
ravdess    tension  144  0.979538 0.951389 1.000000           0.746429  0.780376
    ALL  stability 1651  0.876157 0.860690 0.891581           0.877384  0.872312
  crema  stability  947  0.868037 0.84

In [ ]:
for state in STATE_NAMES:
    print(f"--- {state}: phân phối nhãn true theo dataset ---")
    print(pd.crosstab(test_out["dataset"], test_out["y_true"][state]))
    print()

--- activation: phân phối nhãn true theo dataset ---
col_0      0    1    2
row_0                 
crema    270  350  327
iemocap  120  211  229
ravdess  138    3    3

--- tension: phân phối nhãn true theo dataset ---
col_0      0    1    2
row_0                 
crema    396  335  216
iemocap  329  154   77
ravdess    0    4  140

--- stability: phân phối nhãn true theo dataset ---
col_0      0    1    2
row_0                 
crema    226  335  386
iemocap   61  148  351
ravdess  144    0    0



In [ ]:
summary = per_dataset_df.pivot(index="dataset", columns="state", values="accuracy")
summary = summary.reindex(["ALL"] + sorted(set(test_out["dataset"])))
summary.to_csv(OUT_DIR / "per_dataset_accuracy_summary.csv")
summary

state,activation,stability,tension
dataset,,,
ALL,0.822502,0.876157,0.898354
crema,0.832058,0.868037,0.893532
iemocap,0.769821,0.858466,0.885681
ravdess,0.964444,NaN,0.979538
